# Village field boundaries -- inference v5

Field polygons for one village from a boundary GeoJSON, using the fine-tuned models.

**What is new**
* **Model choice per output.** `EXTENT_MODEL` (field vs not-field) and `BOUNDARY_MODEL` (lines between fields) can be different checkpoints, e.g. the old `ft_best` extent with the sharp `ft21` boundaries.
* **Field extraction built for sharp boundary maps.** The fine-tuned boundary map is confident but has small gaps in some bunds. Simple "one field per walled-off area" extraction merges fields through every gap -- that is what made `ft21` look worse than `ft_best` in the fine-tune table. Here walled-off areas are also split where they narrow into a neck (a gap), and splits without boundary evidence are merged back.
* **Calibration on your labelled fields (section 6).** Before the village run, every model combination x extraction setting is scored on the held-out strips of Kurankhed / Borgaon / Sola (the same fields the fine-tune never trained on). The best one is used for the village automatically (`APPLY_CALIBRATION`).
* **Flip test-time augmentation** (`TTA`), straight gap-free polygons with a vertex budget, tree filling and dense-crop rescue (as in v3/v4).

**Run:** Runtime -> GPU, set the CONFIG cell, Run all. Check the boundary preview (section 3) and the calibration table (section 6).

**Outputs** (`OUT_DIR`): `*_fields.gpkg` (UTM 43N), `*_fields.geojson` (WGS84), `*_village_boundary.geojson`, `*_imagery.tif`, `*_preview.png`, `*_run_log.json`, `calibration.csv`.

In [ ]:
# 1. CONFIG
from google.colab import drive
drive.mount('/content/drive')

# --- village -----------------------------------------------------------------
VILLAGE_BOUNDARY_PATH = "/content/drive/MyDrive/CCSB/borgaon_manju.json"
VILLAGE_NAME   = None             # None = taken from the file name
SITE_NAME      = None             # used in output filenames; None = taken from VILLAGE_NAME
FEATURE_FILTER = None             # None = union of all features; or e.g. {'NAME': 'Borgaon Manju'}
BOUNDARY_BUFFER_M    = 0.0
MIN_VILLAGE_HA       = 100
ALLOW_SMALL_BOUNDARY = False
IMAGERY_MARGIN_M     = 150
CLIP_TO_VILLAGE      = True
MAX_PIXELS           = 400_000_000
BLOCK_PX             = 3072
MARGIN_PX            = 256

# --- models ------------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/AI4SF_BD_TF'
MODELS = {                         # name -> checkpoint (missing files are skipped)
    'ft21': f'{DRIVE_ROOT}/finetune/v2/checkpoints/ft21_best.pt',
    'old':  f'{DRIVE_ROOT}/finetune/checkpoints/ft_best.pt',
}
EXTENT_MODEL   = 'ft21'            # field / not-field   (calibration may change these two)
BOUNDARY_MODEL = 'ft21'            # lines between fields
TTA        = True                  # average 4 flips: more continuous boundaries, 4x inference time
USE_FP16   = True
BATCH_SIZE = 16
OVERLAP    = 64

# --- calibration on labelled held-out strips ------------------------------------
RUN_CALIBRATION   = True
APPLY_CALIBRATION = True           # use the best model combination + settings for the village run
CALIB_AREAS = [                    # areas without labels or imagery on Drive are skipped
    dict(name='kurankhed', labels='/content/drive/MyDrive/kurn_ground_truth_4326.geojson',
         imagery=f'{DRIVE_ROOT}/finetune/imagery/finetune_area_1m.tif'),
    dict(name='borgaon', labels='/content/drive/MyDrive/gt_sola.geojson', lat_min=19.0,
         imagery=f'{DRIVE_ROOT}/finetune/v2/imagery/borgaon_1m.tif'),
    dict(name='sola', labels='/content/drive/MyDrive/gt_sola.geojson', lat_max=19.0,
         imagery=f'{DRIVE_ROOT}/finetune/v2/imagery/sola_1m.tif'),
]
HELDOUT_FRACTION    = 0.2          # must match VAL_FRACTION of the fine-tune
EDGE_BUFFER_M       = 5            # trusted region, as in the fine-tune
FILL_HOLES_BELOW_M2 = 2000
CALIB_COMBOS = [('ft21', 'ft21'), ('old', 'old'), ('old', 'ft21')]     # (extent model, boundary model)
CALIB_GRID = dict(SEED_MODE=['neck', 'cores'], CORE_T=[0.2, 0.35], GAP_CLOSE_R=[0, 2], MERGE_T=[0, 0.3])

# --- field extraction (defaults: DEFAULT_SEG in section 2b; values here override them) ---
SEG_OVERRIDES = {}                 # e.g. {'CORE_T': 0.35, 'TREE_FILL_MAX_PX': 0}

# --- polygons ----------------------------------------------------------------
TOPO_TOL_M     = 2.5               # straighten wiggles smaller than this (m)
MAX_VERTICES   = 16                # per-field vertex budget (0 = none); raise to 24-32 for curved fields
MIN_FIELD_M2   = 500
MIN_AREA_FRAC  = 0.10
SMALL_FIELD_HA = 0.10
MAX_FIELD_HA   = 22.0

# --- fixed -------------------------------------------------------------------
UTM_CRS      = 'EPSG:32643'
RES_M        = 1.0
PATCH_PX     = 256
MAX_CHUNK_PX = 1200

import os, re as _re
_stem = os.path.splitext(os.path.basename(VILLAGE_BOUNDARY_PATH))[0]
VILLAGE_NAME = VILLAGE_NAME or _stem
SITE_NAME    = SITE_NAME or _re.sub(r'[^A-Za-z0-9_-]+', '_', VILLAGE_NAME).strip('_').lower()
OUT_DIR      = f'{DRIVE_ROOT}/inference_v5/{SITE_NAME}'

In [ ]:
# 2. SETUP -- install, imports, models
!pip -q install segmentation-models-pytorch

import io, json, time, gc, hashlib, itertools, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

from PIL import Image
from shapely.geometry import shape, Point, Polygon, MultiPolygon
from shapely.ops import unary_union
from shapely.validation import make_valid
from rasterio.features import rasterize
from rasterio.transform import from_origin, Affine
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter
from skimage.segmentation import watershed, expand_labels, find_boundaries
from skimage.morphology import disk, h_maxima
from tqdm.auto import tqdm

import logging
logging.getLogger('rasterio._env').setLevel(logging.ERROR)

os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}  |  smp {smp.__version__}  |  torch {torch.__version__}")
if device.type == 'cpu':
    print("WARNING: no GPU -- a whole village will be very slow. Runtime > Change runtime type > GPU.")


class TripleHeadModel(nn.Module):
    # identical architecture to training; encoder_weights=None because the checkpoint overwrites them
    def __init__(self, encoder_name='efficientnet-b4', encoder_weights=None, dropout_p=0.2):
        super().__init__()
        base = smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights, in_channels=3, classes=1)
        self.encoder, self.decoder = base.encoder, base.decoder
        dc = base.decoder.blocks[-1].conv2[0].out_channels
        self.extent_dropout = nn.Dropout2d(dropout_p)
        self.boundary_dropout = nn.Dropout2d(dropout_p)
        self.extent_head = nn.Conv2d(dc, 1, 3, padding=1)
        self.boundary_head = nn.Conv2d(dc, 1, 3, padding=1)
        self.distance_head = nn.Conv2d(dc, 1, 3, padding=1)

    def forward(self, x):
        d = self.decoder(self.encoder(x))
        return (self.extent_head(self.extent_dropout(d)), self.boundary_head(self.boundary_dropout(d)),
                self.distance_head(d))


LOADED = {}
for name, path in MODELS.items():
    if not os.path.exists(path):
        print(f"  model '{name}' not found, skipped: {path}")
        continue
    m = TripleHeadModel().to(device)
    ck = torch.load(path, map_location=device)
    m.load_state_dict(ck['model'])
    LOADED[name] = m.eval().to(memory_format=torch.channels_last)
    print(f"  model '{name}': {path} (epoch {ck.get('epoch', '?')})")
for role, name in (('EXTENT_MODEL', EXTENT_MODEL), ('BOUNDARY_MODEL', BOUNDARY_MODEL)):
    assert name in LOADED, f"{role} = '{name}' is not loaded -- available: {list(LOADED)}"
torch.backends.cudnn.benchmark = True

In [ ]:
# 2b. FUNCTIONS -- inference, field extraction, polygons (all tested outside Colab)

# --- inference -------------------------------------------------------------------
@torch.no_grad()
def predict(model, img_uint8, tta=None, overlap=None, batch_size=None, fp16=None):
    """Sliding-window extent + boundary probabilities (Hann-weighted overlaps). With TTA the 4 flips
    (none, left-right, up-down, both) are averaged."""
    tta = TTA if tta is None else tta
    overlap = OVERLAP if overlap is None else overlap
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    fp16 = USE_FP16 if fp16 is None else fp16
    img = img_uint8.astype('float32') / 255.0
    H, W = img.shape[1], img.shape[2]
    stride = PATCH_PX - overlap
    ph = (stride - (H - PATCH_PX) % stride) % stride if H > PATCH_PX else PATCH_PX - H
    pw = (stride - (W - PATCH_PX) % stride) % stride if W > PATCH_PX else PATCH_PX - W
    ip = np.pad(img, ((0, 0), (0, ph + PATCH_PX), (0, pw + PATCH_PX)), mode='reflect')
    del img
    Hp, Wp = ip.shape[1], ip.shape[2]
    acc, wsum = np.zeros((2, Hp, Wp), 'float32'), np.zeros((Hp, Wp), 'float32')
    ramp = np.hanning(PATCH_PX)
    win = np.outer(ramp, ramp).astype('float32') + 1e-6
    coords = [(y, x) for y in range(0, Hp - PATCH_PX + 1, stride) for x in range(0, Wp - PATCH_PX + 1, stride)]
    flips = [(), (3,), (2,), (2, 3)] if tta else [()]
    use_amp = fp16 and device.type == 'cuda'
    for i in range(0, len(coords), batch_size):
        bc = coords[i:i + batch_size]
        bi = torch.from_numpy(np.stack([ip[:, y:y + PATCH_PX, x:x + PATCH_PX] for y, x in bc]))
        bi = bi.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
        es, bs = 0, 0
        for f in flips:
            xb = torch.flip(bi, f) if f else bi
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                e, b, _ = model(xb)
            e, b = torch.sigmoid(e.float()), torch.sigmoid(b.float())
            if f:
                e, b = torch.flip(e, f), torch.flip(b, f)
            es, bs = es + e, bs + b
        e = (es / len(flips)).cpu().numpy()[:, 0]
        b = (bs / len(flips)).cpu().numpy()[:, 0]
        for (y, x), ee, bb in zip(bc, e, b):
            acc[0, y:y + PATCH_PX, x:x + PATCH_PX] += ee * win
            acc[1, y:y + PATCH_PX, x:x + PATCH_PX] += bb * win
            wsum[y:y + PATCH_PX, x:x + PATCH_PX] += win
    acc /= np.maximum(wsum, 1e-6)
    return acc[0, :H, :W], acc[1, :H, :W]


def get_probs(img, ext_model=None, bnd_model=None):
    """Extent from one model, boundary from another (or the same: one pass)."""
    ext_model, bnd_model = ext_model or EXTENT_MODEL, bnd_model or BOUNDARY_MODEL
    ep, bp = predict(LOADED[ext_model], img)
    if bnd_model != ext_model:
        _, bp = predict(LOADED[bnd_model], img)
    return ep, bp


# --- raster helpers (raster_v2) ---------------------------------------------------
def drop_small_regions(labels, min_px):
    counts = np.bincount(labels.ravel())
    small = counts < min_px
    small[0] = False
    labels[small[labels]] = 0
    return labels


def fill_small_holes(mask, max_hole_px):
    """binary_fill_holes, but only for holes up to max_hole_px (None = all holes): a tree inside a
    field is filled, a settlement or pond surrounded by fields is not."""
    if max_hole_px is None:
        return ndi.binary_fill_holes(mask)
    holes = ndi.binary_fill_holes(mask) & ~mask
    lab, n = ndi.label(holes)
    if n == 0:
        return mask
    small = np.bincount(lab.ravel()) <= max_hole_px
    small[0] = False
    return mask | small[lab]


def normalize_boundary(bp, mask=None, pctl=99, floor=0.15):
    """Rescale the boundary map so its pctl-th percentile (inside mask) becomes 1. The fine-tuned
    boundary head is under-confident (peaks ~0.45, never > 0.5), so fixed thresholds such as
    'boundary < 0.30' would mean something different on every checkpoint. floor caps the gain
    (at most 1/floor) so a block with almost no boundaries does not get its noise blown up."""
    if pctl is None:
        return bp
    v = bp[mask] if mask is not None and mask.any() else bp
    p = float(np.percentile(v, pctl)) if v.size else 1.0
    return np.clip(bp / max(p, floor), 0, 1).astype(np.float32)


def make_seeds(extent_mask, bp, dist, mode='neck_dist', seed_bnd_max=0.30, seed_dist_min=0.25,
               seed_min_px=30, seed_open_r=1, neck_h=2.0):
    """Seed markers, one per field.
    'cores'    : pixels away from boundaries AND with high predicted distance (v2/v3 rule). Best when
                 the distance head is good; merges fields when it is weak.
    'neck'     : areas enclosed by the boundary map, split wherever they narrow into a neck (a gap
                 in a bund) -- peaks of the area's own distance-to-boundary that stand >= neck_h px
                 above the neck between them. Does not use the distance head.
    'neck_dist': as 'neck', with the peaks weighted by the predicted distance.
    Over-splits from the neck modes are undone by merge_weak_boundaries, which keeps a split only
    when the shared border has boundary evidence."""
    bcore = extent_mask & (bp < seed_bnd_max)
    if mode == 'cores':
        sm = ndi.binary_opening(bcore & (dist > seed_dist_min), structure=disk(1))
        markers, _ = ndi.label(sm)
        return drop_small_regions(markers, seed_min_px)
    if mode not in ('neck', 'neck_dist'):
        raise ValueError(f"unknown seed mode {mode!r}")
    if seed_open_r > 0:
        bcore = ndi.binary_opening(bcore, structure=disk(seed_open_r))
    mb, _ = ndi.label(bcore)
    mb = drop_small_regions(mb, seed_min_px)
    geo = ndi.distance_transform_edt(mb > 0).astype(np.float32)
    surf = geo * (0.5 + dist) if mode == 'neck_dist' else geo
    peaks = h_maxima(gaussian_filter(surf, 1.0), neck_h) > 0
    markers, _ = ndi.label(ndi.binary_dilation(peaks, structure=disk(1)) & (mb > 0))
    # a boundary core with no peak at all (tiny / odd shape) still gets one seed
    has = np.zeros(int(mb.max()) + 1, bool)
    has[np.unique(mb[markers > 0])] = True
    orphan = (mb > 0) & ~has[mb]
    if orphan.any():
        ol, _ = ndi.label(orphan)
        markers = np.where(ol > 0, ol + markers.max(), markers)
    return markers.astype(np.int32)


def merge_weak_boundaries(labels, boundary_prob, thresh=0.30, min_contact_px=10,
                          max_area_px=None):
    """Region-adjacency merge: two touching regions are merged when the boundary probability
    along their shared border is weak (mean < thresh). Undoes over-segmentation and any split
    that has no boundary evidence behind it. Weakest borders are merged first."""
    if labels.max() == 0:
        return labels, 0
    L, B = labels, boundary_prob
    los, his, vals = [], [], []
    for a, b, va, vb in ((L[:, :-1], L[:, 1:], B[:, :-1], B[:, 1:]),
                         (L[:-1, :], L[1:, :], B[:-1, :], B[1:, :])):
        m = (a != b) & (a > 0) & (b > 0)
        los.append(np.minimum(a[m], b[m])); his.append(np.maximum(a[m], b[m]))
        vals.append(np.maximum(va[m], vb[m]))
    lo, hi, v = (np.concatenate(x) for x in (los, his, vals))
    if lo.size == 0:
        return labels, 0
    n = int(L.max()) + 1
    key = lo.astype(np.int64) * n + hi
    uniq, inv = np.unique(key, return_inverse=True)
    cnt = np.bincount(inv)
    mean = np.bincount(inv, weights=v) / cnt
    cand = np.where((cnt >= min_contact_px) & (mean < thresh))[0]
    cand = cand[np.argsort(mean[cand])]

    parent = np.arange(n)
    area = np.bincount(L.ravel(), minlength=n).astype(np.int64)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    merged = 0
    for c in cand:
        ra, rb = find(int(uniq[c] // n)), find(int(uniq[c] % n))
        if ra == rb:
            continue
        if max_area_px is not None and area[ra] + area[rb] > max_area_px:
            continue
        parent[rb] = ra
        area[ra] += area[rb]
        merged += 1
    root = np.array([find(i) for i in range(n)])
    root[0] = 0
    return root[L].astype(L.dtype), merged


def smooth_labels(labels, radius=2):
    """Raster 'dagger' removal that keeps the partition: open each region with a disk, then
    give the removed spike pixels to the nearest surviving neighbour."""
    if radius <= 0 or labels.max() == 0:
        return labels
    fp = disk(radius)
    out = np.zeros_like(labels)
    for i, sl in enumerate(ndi.find_objects(labels), start=1):
        if sl is None:
            continue
        sl = tuple(slice(max(s.start - radius, 0), s.stop + radius) for s in sl)
        m = labels[sl] == i
        o = ndi.binary_opening(m, structure=fp)
        if not o.any():                     # region thinner than the disk everywhere: keep as is
            o = m
        out[sl][o] = i
    fg = labels > 0
    filled = expand_labels(out, distance=radius + 1)
    out = np.where(fg & (out == 0), filled, out)
    out[~fg] = 0
    return out


def smooth_labels_gaussian(labels, sigma=1.5, valid=None):
    """Smooth every region's outline without breaking the tiling: blur each region's mask
    (and the background's) with a Gaussian and give each pixel to the highest blurred value.
    Removes wiggles shorter than ~3*sigma px; corners round by ~sigma px."""
    if sigma <= 0 or labels.max() == 0:
        return labels
    m = int(np.ceil(3 * sigma))
    best = gaussian_filter((labels == 0).astype(np.float32), sigma)
    out = np.zeros_like(labels)
    for i, sl in enumerate(ndi.find_objects(labels), start=1):
        if sl is None:
            continue
        sl = tuple(slice(max(s.start - m, 0), s.stop + m) for s in sl)
        b = gaussian_filter((labels[sl] == i).astype(np.float32), sigma)
        bv, ov = best[sl], out[sl]
        upd = b > bv
        bv[upd] = b[upd]
        ov[upd] = i
    if valid is not None:
        out[~valid] = 0
    return out


def rescue_compact_patches(labels, boundary_prob, valid, img=None, min_px=2000, max_px=220_000,
                           min_solidity=0.88, min_width_px=20, min_exg=0.02, bnd_cut=0.5, open_r=2):
    """Fields the model scored as non-crop (dense sugarcane / banana / orchard blocks look like
    trees): a leftover non-field patch that is big, compact and field-shaped, and green, becomes a
    field. Returns (labels, list of new label ids) so the caller can flag them for review."""
    from skimage.measure import regionprops
    bg = (labels == 0) & valid & (boundary_prob < bnd_cut)        # cut along predicted boundaries
    if open_r > 0:
        bg = ndi.binary_opening(bg, structure=disk(open_r))      # detach roads / bunds
    comp, n = ndi.label(bg)
    if n == 0:
        return labels, []
    exg = None
    if img is not None and min_exg is not None:
        f = img.astype(np.float32)
        s = f.sum(0) + 1e-6
        exg = (2 * f[1] - f[0] - f[2]) / s                       # excess-green index, -1..2
    out = labels.copy()
    nid = int(labels.max()) + 1
    new_ids = []
    for p in regionprops(comp):
        if not (min_px <= p.area <= max_px):
            continue
        if p.solidity < min_solidity or p.minor_axis_length < min_width_px:
            continue
        if exg is not None:
            rr, cc = p.coords[:, 0], p.coords[:, 1]
            if float(exg[rr, cc].mean()) < min_exg:              # bare / built-up / water: not a crop
                continue
        out[p.coords[:, 0], p.coords[:, 1]] = nid
        new_ids.append(nid)
        nid += 1
    return out, new_ids


def fill_small_gaps(labels, valid, max_px=1500, min_enclosed=0.6, bund_r=2):
    """Trees, bushes and shadows: a small non-field blob inside the village that is mostly
    surrounded by fields is given to the surrounding field(s) -- each pixel to the nearest field,
    so a tree on a bund is split between the two fields, a tree inside a field goes to that field.
    The non-field mask is first opened with a disk of bund_r px so that thin bunds (< ~2*bund_r+1 px)
    no longer connect every tree into one huge network."""
    bg = (labels == 0) & valid
    blobs = ndi.binary_opening(bg, structure=disk(bund_r)) if bund_r > 0 else bg
    comp, n = ndi.label(blobs)
    if n == 0:
        return labels, 0
    sizes = np.bincount(comp.ravel(), minlength=n + 1)
    dil = ndi.grey_dilation(comp, size=3)
    ring = (dil > 0) & (comp == 0)                               # pixels just outside each blob
    rc = dil[ring]
    # enclosed = ring is field or a thin bund, not another blob and not outside the village
    enclosed = (labels[ring] > 0) | (bg[ring] & ~blobs[ring])
    tot = np.bincount(rc, minlength=n + 1).astype(float)
    enc = np.bincount(rc, weights=enclosed.astype(float), minlength=n + 1)
    frac = enc / np.maximum(tot, 1)
    ok = (sizes <= max_px) & (frac >= min_enclosed)
    ok[0] = False
    if not ok.any():
        return labels, 0
    out = labels.copy()
    m = int(np.ceil(np.sqrt(max_px))) + 2 * bund_r + 2
    slices = ndi.find_objects(comp)
    grow = disk(bund_r) if bund_r > 0 else None
    for k in np.nonzero(ok)[0]:
        sl = slices[k - 1]
        sl = tuple(slice(max(s.start - m, 0), s.stop + m) for s in sl)
        tgt = comp[sl] == k
        if grow is not None:                                     # give back what the opening shaved off
            tgt = ndi.binary_dilation(tgt, structure=grow) & bg[sl]
        sub = labels[sl]
        if not (sub > 0).any():
            continue
        near = expand_labels(sub, distance=2 * m)
        out[sl][tgt] = near[tgt]
    return out, int(ok.sum())


# --- scoring helpers (fine-tune) ----------------------------------------------------
def quick_instances(extent_prob, boundary_prob, extent_thresh=0.5, bnd_pctl=99, core_max=0.3, min_px=200):
    """Small, fixed instance extraction used ONLY to compare checkpoints field-by-field (not the
    production pipeline): boundary rescaled per window, cores = extent & low boundary, watershed."""
    ext = extent_prob > extent_thresh
    p = np.percentile(boundary_prob[ext], bnd_pctl) if ext.any() else 1.0
    bp = np.clip(boundary_prob / max(p, 0.15), 0, 1)
    cores = ndi.binary_opening(ext & (bp < core_max), structure=np.ones((3, 3), bool))
    markers, _ = ndi.label(cores)
    lab = watershed(bp, markers, mask=ext)
    cnt = np.bincount(lab.ravel())
    small = cnt < min_px
    small[0] = False
    lab[small[lab]] = 0
    return lab


def match_fields(gt_inst, pred, valid=None, iou_thr=0.5):
    """Field-level scores. A GT field counts only if it lies (>= 95 %) inside `valid`.
    matched  = GT fields with a predicted field at IoU >= iou_thr (one-to-one by construction
               for iou_thr >= 0.5)
    merged   = predicted fields that cover >= 50 % of two or more GT fields
    split    = GT fields cut into 2+ predicted pieces that each cover >= 20 % of it"""
    g = gt_inst.ravel().astype(np.int64)
    p = pred.ravel().astype(np.int64)
    ga = np.bincount(g, minlength=int(g.max()) + 1)
    if valid is not None:
        gv = np.bincount(g, weights=valid.ravel().astype(float), minlength=len(ga))
        keep = (ga > 0) & (gv >= 0.95 * ga)
    else:
        keep = ga > 0
    keep[0] = False
    ids = np.nonzero(keep)[0]
    if len(ids) == 0:
        return dict(n_gt=0, matched=0, recall=0.0, mean_iou=0.0, merged=0, split=0)
    m = (g > 0) & (p > 0) & keep[g]
    n = int(p.max()) + 1
    u, c = np.unique(g[m] * n + p[m], return_counts=True)
    gi, pi = u // n, u % n
    pa = np.bincount(p, minlength=n)
    iou = c / (ga[gi] + pa[pi] - c)
    best = np.zeros(len(ga))
    np.maximum.at(best, gi, iou)
    frac_g = c / ga[gi]
    merged = int((np.bincount(pi[frac_g >= 0.5], minlength=n) >= 2).sum())
    split = int((np.bincount(gi[frac_g >= 0.2], minlength=len(ga)) >= 2).sum())
    b = best[ids]
    return dict(n_gt=len(ids), matched=int((b >= iou_thr).sum()), recall=float((b >= iou_thr).mean()),
                mean_iou=float(b.mean()), merged=merged, split=split)


# --- field extraction + calibration helpers (infer_seg) ------------------------------
DEFAULT_SEG = dict(
    EXT_T=0.5,            # extent probability above this = field land
    EDGE_SIGMA=0.7,       # px blur on the boundary map
    BND_NORM=99,          # rescale boundary so this percentile (inside the village) = 1; None = off
    CORE_T=0.2,           # boundary (rescaled) at/above this is a wall between fields
    GAP_CLOSE_R=0,        # px; thicken walls before seeding so gaps up to ~2*R px in a bund still separate fields
    SEED_MODE='neck',     # 'neck': split walled-off areas where they narrow (a gap in a bund) | 'cores': one field per walled-off area
    NECK_H=2.0,           # 'neck' only: px a neck must be narrower than the cores beside it
    SEED_MIN_PX=30,
    MERGE_T=0.3,          # merge touching fields whose shared border averages below this (rescaled); 0 = off
    MIN_REGION_PX=200,
    EXTENT_HOLE_MAX_PX=1500,
    MAX_FIELD_PX=220_000,
    RESCUE=True, RESCUE_MIN_PX=2000, RESCUE_MIN_SOLIDITY=0.88, RESCUE_MIN_WIDTH_PX=20, RESCUE_MIN_EXG=0.02,
    TREE_FILL_MAX_PX=1500, TREE_MIN_ENCLOSED=0.6, BUND_OPEN_PX=2,
    SMOOTH_R=1, GAUSS_SIGMA=1.0,
)


def segment(ep, bp, vm, img=None, post=True, **cfg):
    """Model outputs -> field label image.
    ep, bp: extent / boundary probability (H, W); vm: bool mask of the area to label; img: RGB uint8
    (3, H, W), used only by the rescue of dense-crop fields. post=False skips rescue / tree fill /
    smoothing (faster; used by the calibration sweep). Returns (labels, rescued_ids, info)."""
    c = {**DEFAULT_SEG, **cfg}
    ext = fill_small_holes(ndi.binary_closing(ep > c['EXT_T'], structure=disk(1)), c['EXTENT_HOLE_MAX_PX']) & vm
    b = bp.astype(np.float32)
    if c['EDGE_SIGMA']:
        b = gaussian_filter(b, c['EDGE_SIGMA'])
    b = normalize_boundary(b, vm, c['BND_NORM'])
    wall = b >= c['CORE_T']
    if c['GAP_CLOSE_R'] > 0:
        wall = ndi.binary_dilation(wall, structure=disk(int(c['GAP_CLOSE_R'])))
    free = ext & ~wall
    if c['SEED_MODE'] == 'cores':
        m, _ = ndi.label(ndi.binary_opening(free, structure=disk(1)))
        markers = drop_small_regions(m, c['SEED_MIN_PX'])
    elif c['SEED_MODE'] == 'neck':
        markers = make_seeds(free, b, np.zeros_like(b), mode='neck', seed_bnd_max=2.0,
                             seed_min_px=c['SEED_MIN_PX'], seed_open_r=1, neck_h=c['NECK_H'])
    else:
        raise ValueError(f"unknown SEED_MODE {c['SEED_MODE']!r}")
    n_seeds = int(len(np.unique(markers)) - 1)
    labels = watershed(b, markers, mask=ext).astype(np.int32)
    # field land that got no seed at all (e.g. a narrow strip entirely under the boundary band)
    ol, _ = ndi.label(ext & (labels == 0))
    cnt = np.bincount(ol.ravel()); cnt[0] = 0
    keep = cnt >= c['MIN_REGION_PX']
    if keep.any():
        ol[~keep[ol]] = 0
        labels = np.where(ol > 0, ol + labels.max(), labels).astype(np.int32)
    n_merged = 0
    if c['MERGE_T'] and c['MERGE_T'] > 0:
        labels, n_merged = merge_weak_boundaries(labels, b, c['MERGE_T'], max_area_px=c['MAX_FIELD_PX'])
    rescued, n_filled = [], 0
    if post:
        if c['RESCUE']:
            labels, rescued = rescue_compact_patches(
                labels, b, vm, img, min_px=c['RESCUE_MIN_PX'], max_px=c['MAX_FIELD_PX'],
                min_solidity=c['RESCUE_MIN_SOLIDITY'], min_width_px=c['RESCUE_MIN_WIDTH_PX'],
                min_exg=c['RESCUE_MIN_EXG'])
        if c['TREE_FILL_MAX_PX'] > 0:
            labels, n_filled = fill_small_gaps(labels, vm, max_px=c['TREE_FILL_MAX_PX'],
                                               min_enclosed=c['TREE_MIN_ENCLOSED'], bund_r=int(c['BUND_OPEN_PX']))
        if c['SMOOTH_R'] > 0:
            labels = smooth_labels(labels, int(c['SMOOTH_R']))
        if c['GAUSS_SIGMA'] > 0:
            labels = smooth_labels_gaussian(labels, c['GAUSS_SIGMA'], valid=vm)
    labels = drop_small_regions(labels, c['MIN_REGION_PX'])
    info = dict(seeds=n_seeds, merged=int(n_merged), rescued=len(rescued), filled=int(n_filled),
                fields=int(len(np.unique(labels)) - 1))
    return labels, rescued, info


def config_grid(grid):
    """{'CORE_T': [0.2, 0.4], ...} -> list of setting dicts (all combinations)."""
    keys = list(grid)
    return [dict(zip(keys, v)) for v in itertools.product(*(grid[k] for k in keys))]


def strip_split_x(inst, frac=0.2):
    """Column of the held-out cut: ~frac of the area's fields (by centroid) lie east of it -- the same
    rule the fine-tune used (geographic_split), so calibration only scores fields it never trained on."""
    cols = np.arange(inst.shape[1])[None, :].repeat(inst.shape[0], 0)
    cnt = np.bincount(inst.ravel())[1:]
    cx = np.bincount(inst.ravel(), weights=cols.ravel())[1:]
    cx = cx[cnt > 0] / cnt[cnt > 0]
    return int(np.ceil(np.quantile(cx, 1 - frac)))


def strip_gt(inst, split_x):
    """GT fields lying entirely east of split_x (fields cut by the line are ignored)."""
    s = inst[:, split_x:]
    return np.where(np.isin(s, np.unique(inst[:, :split_x])), 0, s)


def score_combo(areas, probs, ext_model, bnd_model, cfg=None, method='segment'):
    """areas: [dict(name, gt, valid)] on the held-out strips; probs: {(area, model): (ep, bp)} on the
    same strips. method 'segment' (this notebook) or 'quick' (the fine-tune's Section 8 extraction).
    Returns (pooled recall, total merged, total split, {area: match_fields result})."""
    per = {}
    for a in areas:
        ep = probs[(a['name'], ext_model)][0]
        bp = probs[(a['name'], bnd_model)][1]
        if method == 'quick':
            lab = quick_instances(ep, bp)
        else:
            lab = segment(ep, bp, np.ones(ep.shape, bool), post=False, **(cfg or {}))[0]
        per[a['name']] = match_fields(a['gt'], lab, a['valid'])
    n = sum(r['n_gt'] for r in per.values())
    pooled = sum(r['matched'] for r in per.values()) / max(n, 1)
    return (float(pooled), int(sum(r['merged'] for r in per.values())),
            int(sum(r['split'] for r in per.values())), per)


# --- topology-preserving polygons (topo_vec) ----------------------------------------
_E, _W, _S, _N = (0, 1), (0, -1), (1, 0), (-1, 0)


def trace_arcs(L):
    """L: 2-D int label array (0 = background). Returns (arcs, P) where each arc is a dict
    {'pts': [(row, col), ...] corner coords, 'left': label, 'right': label, 'closed': bool}.
    'left' is the label on the left when walking the arc in screen orientation (row down)."""
    H, W = L.shape
    P = np.zeros((H + 2, W + 2), dtype=np.int64)
    P[1:-1, 1:-1] = L
    R, C = H + 3, W + 3                                   # corner grid size
    # hcr[i, j]: crack from corner (i, j) to (i, j+1), between P[i-1, j] (above) and P[i, j] (below)
    hcr = np.zeros((R, C), bool)
    hcr[1:H + 2, 0:W + 2] = P[0:H + 1, :] != P[1:H + 2, :]
    # vcr[i, j]: crack from corner (i, j) to (i+1, j), between P[i, j-1] (left) and P[i, j] (right)
    vcr = np.zeros((R, C), bool)
    vcr[0:H + 2, 1:W + 2] = P[:, 0:W + 1] != P[:, 1:W + 2]

    deg = np.zeros((R, C), np.int8)
    deg += hcr                                            # E
    deg[:, 1:] += hcr[:, :-1]                             # W
    deg += vcr                                            # S
    deg[1:, :] += vcr[:-1, :]                             # N
    node = deg >= 3

    Pl = P.tolist()
    hv = np.zeros((R, C), bool)                           # visited
    vv = np.zeros((R, C), bool)
    hl, vl = hcr.tolist(), vcr.tolist()
    hvl, vvl, nodel = hv.tolist(), vv.tolist(), node.tolist()

    def step(i, j, d):
        """crack exists? -> (visited-list, ci, cj, next corner, left, right)"""
        if d == _E:
            return hl[i][j], hvl, i, j, (i, j + 1), Pl[i - 1][j], Pl[i][j]
        if d == _W:
            return hl[i][j - 1], hvl, i, j - 1, (i, j - 1), Pl[i][j - 1], Pl[i - 1][j - 1]
        if d == _S:
            return vl[i][j], vvl, i, j, (i + 1, j), Pl[i][j], Pl[i][j - 1]
        return vl[i - 1][j], vvl, i - 1, j, (i - 1, j), Pl[i - 1][j - 1], Pl[i - 1][j]

    def crack_ok(i, j, d):
        if d == _E: return 0 <= j < C - 1 and hl[i][j]
        if d == _W: return j >= 1 and hl[i][j - 1]
        if d == _S: return 0 <= i < R - 1 and vl[i][j]
        return i >= 1 and vl[i - 1][j]

    def walk(i0, j0, d0, closed_start=False):
        pts = [(i0, j0)]
        _, vis, ci, cj, nxt, left, right = step(i0, j0, d0)
        vis[ci][cj] = True
        d = d0
        i, j = nxt
        while True:
            pts.append((i, j))
            if nodel[i][j] or (closed_start and (i, j) == (i0, j0)):
                break
            back = (-d[0], -d[1])
            for nd in (d, (-d[1], d[0]), (d[1], -d[0])):    # straight, then the two turns
                if nd == back or not crack_ok(i, j, nd):
                    continue
                ok, vis, ci, cj, nxt2, _, _ = step(i, j, nd)
                if vis[ci][cj]:
                    continue
                vis[ci][cj] = True
                d = nd
                i, j = nxt2
                break
            else:                                         # dead end cannot happen on a valid crack graph
                break
        pts = [(a - 1, b - 1) for a, b in pts]          # padded-grid corners -> corners of L
        return {'pts': pts, 'left': left, 'right': right, 'closed': pts[0] == pts[-1]}

    arcs = []
    ni, nj = np.nonzero(node)
    for i, j in zip(ni.tolist(), nj.tolist()):
        for d in (_E, _W, _S, _N):
            if not crack_ok(i, j, d):
                continue
            _, vis, ci, cj, _, _, _ = step(i, j, d)
            if not vis[ci][cj]:
                arcs.append(walk(i, j, d))
    # closed loops with no node (a field surrounded by one other label / background)
    for arr, vis, d in ((hcr, hvl, _E), (vcr, vvl, _S)):
        ii, jj = np.nonzero(arr)
        for i, j in zip(ii.tolist(), jj.tolist()):
            if not vis[i][j]:
                arcs.append(walk(i, j, d, closed_start=True))
    return arcs


def _drop_collinear(pts):
    """Remove interior points lying on a straight run (keeps endpoints)."""
    a = np.asarray(pts, dtype=np.float64)
    if len(a) <= 2:
        return a
    d1 = a[1:-1] - a[:-2]
    d2 = a[2:] - a[1:-1]
    cross = d1[:, 0] * d2[:, 1] - d1[:, 1] * d2[:, 0]
    keep = np.r_[True, cross != 0, True]
    return a[keep]


def _dp_importance(a, floor):
    """Douglas-Peucker importance for each point of open polyline a (endpoints = inf).
    Importance is made monotone (child <= parent) so thresholding keeps a valid DP subset.
    Recursion stops below `floor` (those points get 0)."""
    n = len(a)
    imp = np.zeros(n)
    imp[0] = imp[-1] = np.inf
    stack = [(0, n - 1, np.inf)]
    while stack:
        s, e, parent = stack.pop()
        if e - s < 2:
            continue
        seg = a[e] - a[s]
        L = np.hypot(seg[0], seg[1])
        q = a[s + 1:e] - a[s]
        if L == 0:
            dist = np.hypot(q[:, 0], q[:, 1])
        else:
            dist = np.abs(q[:, 0] * seg[1] - q[:, 1] * seg[0]) / L
        k = int(np.argmax(dist))
        dk = float(dist[k])
        if dk < floor:
            continue
        k += s + 1
        imp[k] = min(dk, parent)
        stack.append((s, k, imp[k]))
        stack.append((k, e, imp[k]))
    return imp


def _chain_rings(arcs):
    """Group half-arcs into closed rings per label using the ORIGINAL geometry.
    Returns {label: [[(arc_idx, reversed), ...], ...]}."""
    chains, halfs = {}, {}
    for idx, a in enumerate(arcs):
        for l, rev in ((a['left'], False), (a['right'], True)):
            if l == 0:
                continue
            if a['closed']:
                chains.setdefault(l, []).append([(idx, rev)])
            else:
                halfs.setdefault(l, []).append((idx, rev))
    for l, hs in halfs.items():
        start_map = {}
        for h in hs:
            p = arcs[h[0]]['p']
            start_map.setdefault(tuple(p[-1]) if h[1] else tuple(p[0]), []).append(h)
        used = set()
        for h0 in hs:
            if h0 in used:
                continue
            chain, h = [], h0
            while h is not None and h not in used:
                used.add(h)
                chain.append(h)
                p = arcs[h[0]]['p']
                end = tuple(p[0]) if h[1] else tuple(p[-1])
                cands = [c for c in start_map.get(end, []) if c not in used]
                if not cands:
                    break
                if len(cands) == 1:
                    h = cands[0]
                    continue
                # several ways out of this corner (diagonal touch): sharpest LEFT turn keeps the
                # region 4-connected, the same rule rasterio.features.shapes uses
                din = np.sign((p[0] - p[1]) if h[1] else (p[-1] - p[-2]))
                left = np.array([-din[1], din[0]])

                def rank(c):
                    q = arcs[c[0]]['p']
                    dout = np.sign((q[-2] - q[-1]) if c[1] else (q[1] - q[0]))
                    return 0 if np.array_equal(dout, left) else (1 if np.array_equal(dout, din) else 2)
                h = min(cands, key=rank)
            chains.setdefault(l, []).append(chain)
    return chains


def vectorize_labels(labels, tol_px=2.0, max_vertices=0):
    """Returns {label: [ring, ...]}; each ring is an (N, 2) float array of (row, col) corner
    coords, closed (first == last). Shells have negative signed_area, holes positive.
    tol_px: Douglas-Peucker tolerance in pixels. max_vertices: per-label vertex budget (0 = none)."""
    arcs = trace_arcs(labels)
    floor = tol_px if tol_px > 0 else 1e-9
    for a in arcs:
        p = _drop_collinear(a['pts'])
        if a['closed']:
            p = p[:-1]
            if len(p) >= 3:
                # open the loop at the point farthest from the start -> two real DP anchors
                k = int(np.argmax(np.hypot(*(p - p[0]).T)))
                i1 = _dp_importance(p[:k + 1], floor)
                i2 = _dp_importance(np.vstack([p[k:], p[:1]]), floor)
                imp = np.r_[i1[:-1], i2[:-1]]
            else:
                imp = np.full(len(p), np.inf)
        else:
            imp = _dp_importance(p, floor)
        a['p'], a['imp'] = p, imp
        a['min_interior'] = 0

    chains = _chain_rings(arcs)
    # a ring made of only 1-2 open arcs would collapse to a line if every arc became straight:
    # such arcs keep at least their most important interior vertex (decided per ARC, so both
    # neighbours still share identical geometry)
    for l, cs in chains.items():
        for ch in cs:
            if len(ch) <= 2:
                for idx, _ in ch:
                    if not arcs[idx]['closed']:
                        arcs[idx]['min_interior'] = 1

    labels_present = set(chains)
    thr = {l: tol_px for l in labels_present}
    if max_vertices and max_vertices > 0:
        for l, cs in chains.items():
            arc_ids = {idx for ch in cs for idx, _ in ch}
            fixed, cand = 0, []
            for idx in arc_ids:
                a = arcs[idx]
                fixed += int(np.isinf(a['imp']).sum()) - (0 if a['closed'] else 1)
                f = a['imp'][np.isfinite(a['imp'])]
                cand.append(f[f >= tol_px])
            v = np.sort(np.concatenate(cand))[::-1] if cand else np.empty(0)
            budget = max(max_vertices - fixed, 0)
            if len(v) > budget:
                thr[l] = max(tol_px, float(v[budget]) + 1e-9)

    for a in arcs:
        t = max(thr.get(a['left'], tol_px), thr.get(a['right'], tol_px))
        keep = a['imp'] >= t
        n_int = int(keep.sum()) - int(np.isinf(a['imp']).sum())
        if a['closed']:
            if keep.sum() < 4:                       # never collapse an island
                keep[np.argsort(-a['imp'])[:min(4, len(keep))]] = True
            q = a['p'][keep]
            a['s'] = np.vstack([q, q[:1]])
        else:
            if n_int < a['min_interior'] and len(a['imp']) > 2:
                inner = a['imp'].copy(); inner[[0, -1]] = -1
                keep[int(np.argmax(inner))] = True
            a['s'] = a['p'][keep]

    out = {}
    for l, cs in chains.items():
        rs = []
        for ch in cs:
            pieces = []
            for idx, rev in ch:
                s = arcs[idx]['s'][::-1] if rev else arcs[idx]['s']
                pieces.append(s if not pieces else s[1:])
            ring = np.vstack(pieces)
            if not np.array_equal(ring[0], ring[-1]):
                ring = np.vstack([ring, ring[:1]])
            if len(ring) >= 4 and abs(signed_area(ring)) > 1e-9:
                rs.append(ring)
        out[l] = rs
    return out


def signed_area(ring):
    """Shoelace in (x=col, y=row) coords. Shells (label on the left, row-down screen) are negative."""
    x, y = ring[:, 1], ring[:, 0]
    return 0.5 * float(np.sum(x[:-1] * y[1:] - x[1:] * y[:-1]))


def rings_to_world(ring, transform):
    """(row, col) corner coords -> (x, y) map coords with an affine (a, b, c, d, e, f)."""
    a, b, c, d, e, f = transform[:6]
    r, cc = ring[:, 0], ring[:, 1]
    return np.column_stack([c + cc * a + r * b, f + cc * d + r * e])


def polygonize_labels_topo(labels, transform, crs, tol_m=TOPO_TOL_M, max_vertices=MAX_VERTICES):
    """Label raster -> simplified, gap-free polygons (one row per polygon part, column 'label')."""
    rings = vectorize_labels(labels, tol_px=tol_m / abs(transform.a), max_vertices=max_vertices)
    aff = (transform.a, transform.b, transform.c, transform.d, transform.e, transform.f)
    geoms, labs = [], []
    for lab, rs in rings.items():
        shells = [_as_polygonal(Polygon(rings_to_world(r, aff))) for r in rs if signed_area(r) < 0]
        holes = [_as_polygonal(Polygon(rings_to_world(r, aff))) for r in rs if signed_area(r) > 0]
        shells = [g for g in shells if g is not None]
        holes = [g for g in holes if g is not None]
        if not shells:
            continue
        try:
            g = unary_union(shells)
            if holes:
                g = g.difference(unary_union(holes))
        except Exception:                                  # GEOS topology error: keep the shells only
            g = unary_union([s.buffer(0) for s in shells])
        g = _as_polygonal(g)
        if g is None:
            continue
        for part in (g.geoms if g.geom_type == 'MultiPolygon' else [g]):
            if part.area > 0:
                geoms.append(part); labs.append(int(lab))
    return gpd.GeoDataFrame({'label': labs}, geometry=geoms, crs=crs)


SEG = {**DEFAULT_SEG, **SEG_OVERRIDES}
print("field extraction settings:", SEG)

In [ ]:
# 3. VILLAGE BOUNDARY -- read from the GeoJSON given in VILLAGE_BOUNDARY_PATH

def _as_polygonal(geom):
    """Valid Polygon/MultiPolygon, or None."""
    if geom is None or geom.is_empty:
        return None
    if not geom.is_valid:
        geom = make_valid(geom)
    if geom.geom_type in ('Polygon', 'MultiPolygon'):
        return geom
    parts = [p for p in (_as_polygonal(g) for g in getattr(geom, 'geoms', [])) if p is not None]
    return unary_union(parts) if parts else None


def load_village_boundary(path, feature_filter=None):
    """Read the boundary file -> (geometry in UTM_CRS, description)."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"VILLAGE_BOUNDARY_PATH not found: {path}")
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        print("  NOTE: file has no CRS -- assuming EPSG:4326 (WGS84 lon/lat), the GeoJSON default.")
        gdf = gdf.set_crs('EPSG:4326')
    n_raw = len(gdf)

    if feature_filter:
        for key, val in feature_filter.items():
            if key not in gdf.columns:
                raise KeyError(f"FEATURE_FILTER key '{key}' not in file attributes: {list(gdf.columns)}")
            gdf = gdf[gdf[key].astype(str).str.strip().str.lower() == str(val).strip().lower()]
        if gdf.empty:
            raise ValueError(f"FEATURE_FILTER {feature_filter} matched none of the {n_raw} features in the file.")
        print(f"  FEATURE_FILTER matched {len(gdf)} of {n_raw} features.")

    gdf = gdf.to_crs(UTM_CRS)
    gdf['geometry'] = gdf.geometry.apply(_as_polygonal)
    gdf = gdf[gdf.geometry.notna()]
    if gdf.empty:
        raise ValueError(f"no polygon features in {path}")

    if len(gdf) > 1:
        print(f"  {len(gdf)} polygon features in the file -> using their UNION as the village. "
              f"If the file holds several villages, set FEATURE_FILTER to select one.")
    geom = _as_polygonal(unary_union(list(gdf.geometry)))
    return geom, f"file: {path} ({len(gdf)} feature{'s' if len(gdf) != 1 else ''})"


village_geom, boundary_desc = load_village_boundary(VILLAGE_BOUNDARY_PATH, FEATURE_FILTER)
boundary_kind = 'file'
village_geom = _as_polygonal(village_geom)
if BOUNDARY_BUFFER_M:
    village_geom = _as_polygonal(village_geom.buffer(BOUNDARY_BUFFER_M))
village_area_ha = village_geom.area / 1e4

# the model was trained in UTM zone 43N; warn if the village is far from it
_c = gpd.GeoSeries([village_geom], crs=UTM_CRS).to_crs('EPSG:4326').iloc[0].centroid
if not (72.0 <= _c.x < 78.0):
    print(f"WARNING: village centroid lon {_c.x:.3f} is outside UTM zone 43 (72E-78E); "
          f"the model was trained in {UTM_CRS}, so results may be distorted.")

print(f"\nvillage         : {VILLAGE_NAME}")
print(f"boundary source : {boundary_desc}")
print(f"village area    : {village_area_ha:.0f} ha ({village_area_ha / 100:.2f} km2)")
if village_area_ha < MIN_VILLAGE_HA and not ALLOW_SMALL_BOUNDARY:
    raise RuntimeError(
        f"Boundary is only {village_area_ha:.1f} ha (< MIN_VILLAGE_HA={MIN_VILLAGE_HA}). That is a fragment, "
        f"not a village -- running on it would give a handful of fields. Fix the boundary file, or set "
        f"ALLOW_SMALL_BOUNDARY=True if it is intentional.")

# --- processing grid: boundary bbox + margin, snapped to whole metres ----------
bminx, bminy, bmaxx, bmaxy = village_geom.bounds
minx = np.floor((bminx - IMAGERY_MARGIN_M) / RES_M) * RES_M
miny = np.floor((bminy - IMAGERY_MARGIN_M) / RES_M) * RES_M
maxx = np.ceil((bmaxx + IMAGERY_MARGIN_M) / RES_M) * RES_M
maxy = np.ceil((bmaxy + IMAGERY_MARGIN_M) / RES_M) * RES_M
w_px, h_px = int(round((maxx - minx) / RES_M)), int(round((maxy - miny) / RES_M))
n_blocks_total = int(np.ceil(w_px / BLOCK_PX) * np.ceil(h_px / BLOCK_PX))
print(f"imagery grid    : {w_px} x {h_px} px ({w_px * h_px / 1e6:.1f} M px) | mosaic in RAM ~{w_px * h_px * 3 / 1e9:.2f} GB "
      f"| up to {n_blocks_total} blocks of {BLOCK_PX} px (empty ones are skipped)")
if w_px * h_px > MAX_PIXELS:
    raise RuntimeError(f"{w_px * h_px:,} px exceeds MAX_PIXELS={MAX_PIXELS:,}. Check the boundary, "
                       f"or raise MAX_PIXELS if you have the RAM.")
if village_area_ha > 3000:
    print(f"NOTE: {village_area_ha:.0f} ha is large for one village -- confirm the boundary file / FEATURE_FILTER "
          f"selected the right polygon(s).")

# --- preview: check this before the long steps ---------------------------------
fig, ax = plt.subplots(figsize=(7, 7))
gpd.GeoSeries([village_geom]).boundary.plot(ax=ax, color='red', linewidth=1.5)
ax.plot([minx, maxx, maxx, minx, minx], [miny, miny, maxy, maxy, miny], 'b--', linewidth=0.8, label='imagery bbox')
ax.set_aspect('equal'); ax.legend(); ax.set_title(f'{VILLAGE_NAME} -- boundary from file, {village_area_ha:.0f} ha')
plt.tight_layout(); plt.show()

In [ ]:
# 4. IMAGERY -- Esri World Imagery, native UTM 1 m/px, parallel fetch + Drive cache

from concurrent.futures import ThreadPoolExecutor

ESRI_URL = 'https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/export'
FETCH_WORKERS = 4


def fetch_chunk(x0, y0, x1, y1, w, h, retries=4):
    params = {'bbox': f'{x0},{y0},{x1},{y1}', 'bboxSR': 32643, 'imageSR': 32643,
              'size': f'{w},{h}', 'format': 'png32', 'f': 'image'}
    for a in range(retries):
        try:
            r = requests.get(ESRI_URL, params=params, timeout=60)
            r.raise_for_status()
            if 'image' not in r.headers.get('Content-Type', ''):
                raise ValueError(f"non-image response: {r.text[:150]}")
            arr = np.array(Image.open(io.BytesIO(r.content)).convert('RGB')).transpose(2, 0, 1)
            if arr.shape != (3, h, w):
                raise ValueError(f"unexpected tile shape {arr.shape}, wanted {(3, h, w)}")
            return arr
        except Exception as e:
            print(f"    retry {a + 1}/{retries}: {e}")
            time.sleep(2 ** a)
    raise RuntimeError("fetch failed")


def fetch_mosaic(minx, maxy, w_px, h_px, res=RES_M, chunk=MAX_CHUNK_PX, workers=FETCH_WORKERS):
    """Pixel-aligned mosaic; (col, row) offsets are from the top-left corner."""
    mosaic = np.zeros((3, h_px, w_px), dtype=np.uint8)
    cols = list(range(0, w_px, chunk)) + [w_px]
    rows = list(range(0, h_px, chunk)) + [h_px]
    jobs = [(c0, c1, r0, r1) for c0, c1 in zip(cols[:-1], cols[1:]) for r0, r1 in zip(rows[:-1], rows[1:])]

    def work(job):
        c0, c1, r0, r1 = job
        arr = fetch_chunk(minx + c0 * res, maxy - r1 * res, minx + c1 * res, maxy - r0 * res,
                          c1 - c0, r1 - r0)
        return job, arr

    with ThreadPoolExecutor(workers) as ex:
        for (c0, c1, r0, r1), arr in tqdm(ex.map(work, jobs), total=len(jobs), desc='fetching'):
            mosaic[:, r0:r1, c0:c1] = arr
            if arr.std() < 1.0:
                print(f"  WARNING: chunk cols {c0}-{c1}, rows {r0}-{r1} is uniform "
                      f"(likely 'no imagery' placeholder)")
    return mosaic


transform = from_origin(minx, maxy, RES_M, RES_M)
raster_path = f'{OUT_DIR}/{SITE_NAME}_imagery.tif'

mosaic = None
if os.path.exists(raster_path):
    with rasterio.open(raster_path) as src:
        if ((src.width, src.height) == (w_px, h_px) and src.crs is not None
                and src.crs.to_epsg() == 32643
                and abs(src.bounds.left - minx) < 1e-3 and abs(src.bounds.top - maxy) < 1e-3):
            mosaic = src.read()
            print(f"reusing cached imagery: {raster_path}")

if mosaic is None:
    mosaic = fetch_mosaic(minx, maxy, w_px, h_px)
    with rasterio.open(raster_path, 'w', driver='GTiff', dtype='uint8', count=3,
                       height=h_px, width=w_px, crs=UTM_CRS, transform=transform,
                       compress='deflate', tiled=True) as dst:
        dst.write(mosaic)
    print(f"imagery saved: {raster_path}")
print(f"mosaic: {mosaic.shape}")

## 6. Calibration on labelled held-out strips
Scores every (extent model, boundary model) x extraction setting on the fields east of each area's held-out line (the fine-tune never trained on them). `matched` = GT fields recovered at IoU >= 0.5. The `quick` rows reproduce the fine-tune notebook's Section 8 extraction for comparison.

In [ ]:
# 6. CALIBRATE (optional; ~5-15 min)
def keep_polygons(geom):
    """make_valid can return a collection -- keep the largest polygon part."""
    if geom is None:
        return None
    if geom.geom_type == 'Polygon':
        return geom
    if geom.geom_type in ('MultiPolygon', 'GeometryCollection'):
        polys = [g for g in geom.geoms if g.geom_type == 'Polygon' and g.area > 1.0]
        return max(polys, key=lambda p: p.area) if polys else None
    return None


def clean_labels(path):
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    gdf = gdf.to_crs(UTM_CRS)
    n0 = len(gdf)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    gdf = gdf[gdf.geometry.area > 1.0].copy()
    from shapely import make_valid
    gdf['geometry'] = [make_valid(g) if not g.is_valid else g for g in gdf.geometry]
    gdf['geometry'] = gdf.geometry.apply(keep_polygons)
    gdf = gdf[gdf.geometry.notna()].reset_index(drop=True)
    print(f"  {os.path.basename(path)}: {n0} -> {len(gdf)} polygons after cleaning")
    return gdf[['geometry']]


def trusted_region(area_gdf):
    union = unary_union(area_gdf.geometry.tolist())
    parts = list(union.geoms) if hasattr(union, 'geoms') else [union]
    holes = [Polygon(r) for p in parts for r in p.interiors]
    region = union.buffer(-EDGE_BUFFER_M)
    small = [h for h in holes if h.area < FILL_HOLES_BELOW_M2]
    if small:
        region = unary_union([region] + small)
    return region, len(holes) - len(small)


calib_table, CALIB_BEST = None, None
if RUN_CALIBRATION:
    combos = [c for c in CALIB_COMBOS if c[0] in LOADED and c[1] in LOADED]
    needed = sorted({m for c in combos for m in c})
    calib_areas, calib_probs = [], {}
    for spec in CALIB_AREAS:
        missing = [p for p in (spec['labels'], spec['imagery']) if not os.path.exists(p)]
        if missing:
            print(f"  skipping {spec['name']}: not found {missing}")
            continue
        gt_gdf = clean_labels(spec['labels'])
        if 'lat_min' in spec or 'lat_max' in spec:
            lat = gt_gdf.geometry.centroid.to_crs('EPSG:4326').y
            sel = (lat >= spec.get('lat_min', -90)) & (lat < spec.get('lat_max', 90))
            gt_gdf = gt_gdf[sel.values].reset_index(drop=True)
        with rasterio.open(spec['imagery']) as src:
            a_img, a_t = src.read(), src.transform
        a_inst = rasterize([(g, i + 1) for i, g in enumerate(gt_gdf.geometry)], out_shape=a_img.shape[1:],
                           transform=a_t, fill=0, dtype='int32')
        region, _ = trusted_region(gt_gdf)
        a_valid = rasterize([(region, 1)], out_shape=a_img.shape[1:], transform=a_t, fill=0, dtype='uint8') > 0
        sx = strip_split_x(a_inst, HELDOUT_FRACTION)
        x0 = max(0, sx - 128)                                  # context for the model
        for mname in needed:
            t0 = time.time()
            ep_, bp_ = predict(LOADED[mname], a_img[:, :, x0:])
            calib_probs[(spec['name'], mname)] = (ep_[:, sx - x0:], bp_[:, sx - x0:])
            print(f"  {spec['name']}: predicted strip with '{mname}' ({time.time() - t0:.0f}s)")
        calib_areas.append(dict(name=spec['name'], gt=strip_gt(a_inst, sx), valid=a_valid[:, sx:],
                                img=a_img[:, :, sx:]))
        print(f"  {spec['name']}: {len(np.unique(calib_areas[-1]['gt'])) - 1} held-out fields")
        del a_img, a_inst, a_valid

    if calib_areas:
        grid = config_grid(CALIB_GRID)
        rows = []
        for ext_m, bnd_m in combos:
            for cfg in [None] + grid:
                method = 'quick' if cfg is None else 'segment'
                full = None if cfg is None else {**SEG, **cfg}
                p, mg, sp, per = score_combo(calib_areas, calib_probs, ext_m, bnd_m, full, method)
                rows.append(dict(extent=ext_m, boundary=bnd_m,
                                 setting='quick (fine-tune Sec. 8)' if cfg is None else
                                 ' '.join(f'{k}={v}' for k, v in cfg.items()),
                                 recall=round(p, 3), merged=mg, split=sp,
                                 **{a: f"{per[a]['matched']}/{per[a]['n_gt']}" for a in per}, cfg=cfg))
            print(f"  scored {ext_m} extent + {bnd_m} boundary ({len(grid) + 1} settings)")
        calib_table = pd.DataFrame(rows).sort_values(['recall', 'merged'], ascending=[False, True]).reset_index(drop=True)
        calib_table.drop(columns='cfg').to_csv(f'{OUT_DIR}/calibration.csv', index=False)
        print("\nTOP 12 (recall = share of held-out fields recovered at IoU >= 0.5, pooled over areas)")
        print(calib_table.drop(columns='cfg').head(12).to_string(index=False))
        print("\nfine-tune Section 8 method, for comparison:")
        print(calib_table[calib_table.cfg.isna()].drop(columns='cfg').to_string(index=False))

        CALIB_BEST = calib_table[calib_table.cfg.notna()].iloc[0]
        if APPLY_CALIBRATION:
            EXTENT_MODEL, BOUNDARY_MODEL = CALIB_BEST['extent'], CALIB_BEST['boundary']
            SEG.update(CALIB_BEST['cfg'])
            print(f"\nAPPLIED: extent '{EXTENT_MODEL}', boundary '{BOUNDARY_MODEL}', {CALIB_BEST['cfg']}")

        # visual: GT (red) vs best (yellow), one 500 px crop per area
        fig, axes = plt.subplots(1, len(calib_areas), figsize=(7 * len(calib_areas), 7))
        for ax, a in zip(np.atleast_1d(axes), calib_areas):
            ep_ = calib_probs[(a['name'], CALIB_BEST['extent'])][0]
            bp_ = calib_probs[(a['name'], CALIB_BEST['boundary'])][1]
            lab = segment(ep_, bp_, np.ones(ep_.shape, bool), post=False, **{**SEG, **CALIB_BEST['cfg']})[0]
            rr = int(np.argmax((a['gt'] > 0).sum(1)))
            H_, W_ = a['gt'].shape
            r0 = int(np.clip(rr - 250, 0, max(H_ - 500, 0)))
            sl = (slice(r0, r0 + 500), slice(0, 500))
            over = a['img'][(slice(None),) + sl].transpose(1, 2, 0).copy()
            over[find_boundaries(a['gt'][sl], mode='inner')] = (255, 0, 0)
            over[find_boundaries(lab[sl], mode='inner')] = (255, 255, 0)
            ax.imshow(over); ax.set_title(f"{a['name']}: GT red, predicted yellow"); ax.axis('off')
        plt.tight_layout(); plt.show()

In [ ]:
# 7. RUN THE WHOLE VILLAGE -- block by block (flat memory), resumable, then clip + area rules
def make_blocks(village_mask, block=BLOCK_PX):
    """Core tiles (r0, r1, c0, c1, n_village_px) that contain at least one village pixel."""
    H, W = village_mask.shape
    out = []
    for r0 in range(0, H, block):
        for c0 in range(0, W, block):
            r1, c1 = min(r0 + block, H), min(c0 + block, W)
            n = int(village_mask[r0:r1, c0:c1].sum())
            if n:
                out.append((r0, r1, c0, c1, n))
    return out


village_mask = rasterize([(village_geom, 1)], out_shape=mosaic.shape[1:], transform=transform,
                         fill=0, dtype='uint8').astype(bool)
blocks = make_blocks(village_mask)
px_area = abs(transform.a) * abs(transform.e)
print(f"extent from '{EXTENT_MODEL}', boundary from '{BOUNDARY_MODEL}', TTA={TTA}")
print(f"settings: {SEG}")

_sig = hashlib.md5(json.dumps({
    'seg': SEG, 'ext': EXTENT_MODEL, 'bnd': BOUNDARY_MODEL, 'tta': TTA, 'fp16': USE_FP16, 'overlap': OVERLAP,
    'topo': [TOPO_TOL_M, MAX_VERTICES], 'blocks': [BLOCK_PX, MARGIN_PX],
    'grid': [float(minx), float(maxy), w_px, h_px], 'village_ha': round(float(village_area_ha), 1),
    'ckpt': {n: [MODELS[n], os.path.getmtime(MODELS[n])] for n in {EXTENT_MODEL, BOUNDARY_MODEL}},
}, sort_keys=True, default=str).encode()).hexdigest()[:8]
BLOCK_DIR = f'{OUT_DIR}/blocks_{_sig}'
os.makedirs(BLOCK_DIR, exist_ok=True)


def process_block(r0, r1, c0, c1):
    """Inference + extraction on the block plus a margin; returns polygons whose centre lies in the core."""
    er0, er1 = max(0, r0 - MARGIN_PX), min(h_px, r1 + MARGIN_PX)
    ec0, ec1 = max(0, c0 - MARGIN_PX), min(w_px, c1 + MARGIN_PX)
    img = mosaic[:, er0:er1, ec0:ec1]
    vm = village_mask[er0:er1, ec0:ec1]
    t = transform * Affine.translation(ec0, er0)
    ep, bp = get_probs(img)
    labels, rescued, info = segment(ep, bp, vm, img=img, post=True, **SEG)
    part = polygonize_labels_topo(labels, t, UTM_CRS)
    part['source'] = np.where(part['label'].isin(rescued), 'rescued', 'model')
    left, right = minx + c0 * RES_M, minx + c1 * RES_M
    top, bottom = maxy - r0 * RES_M, maxy - r1 * RES_M
    if len(part):
        rp = part.geometry.representative_point()
        part = part[(rp.x >= left) & (rp.x < right) & (rp.y > bottom) & (rp.y <= top)]
        part = part[part.geometry.notna() & ~part.geometry.is_empty & part.geometry.is_valid]
    return part, info, (img, ep, bp, labels, vm)


sample_idx = int(np.argmax([b[4] for b in blocks]))
sample, parts = None, []
t_run = time.time()
for k, (r0, r1, c0, c1, n_vil) in enumerate(blocks):
    stem = f'{BLOCK_DIR}/b_{r0}_{c0}'
    if os.path.exists(stem + '.gpkg'):
        part = gpd.read_file(stem + '.gpkg'); parts.append(part)
        print(f"[{k + 1}/{len(blocks)}] resumed from cache: {len(part)} polygons"); continue
    if os.path.exists(stem + '.empty'):
        print(f"[{k + 1}/{len(blocks)}] resumed from cache: 0 polygons"); continue
    tb = time.time()
    part, info, arrays = process_block(r0, r1, c0, c1)
    if len(part):
        part.to_file(stem + '.gpkg', driver='GPKG'); parts.append(part)
    else:
        open(stem + '.empty', 'w').close()
    if k == sample_idx:
        sample = arrays
    del arrays
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    print(f"[{k + 1}/{len(blocks)}] rows {r0}-{r1} cols {c0}-{c1} | seeds {info['seeds']} merged {info['merged']} "
          f"rescued {info['rescued']} trees filled {info['filled']} | {len(part)} polygons | {time.time() - tb:.0f}s")
print(f"all blocks done in {(time.time() - t_run) / 60:.1f} min")
if not parts:
    raise RuntimeError("No polygons produced -- check the extent/boundary maps of the busiest block below.")

# --- global stage: area rules, exact clip, attributes ----------------------------------
gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry='geometry', crs=UTM_CRS)
n_blocks_polys = len(gdf)
min_area_m2 = max(MIN_FIELD_M2, gdf.geometry.area.median() * MIN_AREA_FRAC)
gdf = gdf[gdf.geometry.area >= min_area_m2]
n_after_min_area = len(gdf)
if CLIP_TO_VILLAGE:
    gdf['geometry'] = gdf.geometry.intersection(village_geom)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].explode(index_parts=False)
    gdf = gdf[(gdf.geom_type == 'Polygon') & gdf.geometry.is_valid & (gdf.geometry.area >= min_area_m2)]
gdf = gdf.reset_index(drop=True)
gdf['id'] = np.arange(1, len(gdf) + 1)
gdf['village'] = VILLAGE_NAME
gdf['area_m2'] = gdf.geometry.area.round(1)
gdf['area_ha'] = (gdf.geometry.area / 1e4).round(3)
gdf['perimeter_m'] = gdf.geometry.length.round(1)
gdf['n_vertices'] = [len(g.exterior.coords) - 1 for g in gdf.geometry]
gdf['qc'] = np.select([gdf.area_ha < SMALL_FIELD_HA, gdf.area_ha > MAX_FIELD_HA], ['small', 'oversize'], 'ok')
if 'source' not in gdf.columns:
    gdf['source'] = 'model'
gdf['source'] = gdf['source'].fillna('model')
gdf = gdf[['id', 'village', 'area_m2', 'area_ha', 'perimeter_m', 'n_vertices', 'qc', 'source', 'geometry']]
print(f"funnel: {n_blocks_polys} polygons from blocks -> {n_after_min_area} after min-area -> {len(gdf)} after clip")
print("source:", gdf.source.value_counts().to_dict(), "| qc:", gdf.qc.value_counts().to_dict())

# --- QC maps for the busiest block -----------------------------------------------------
if sample is not None:
    img, ep, bp, lab, vm = sample
    step = max(1, int(max(vm.shape) / 1500))
    sl = (slice(None, None, step), slice(None, None, step))
    rgb = img[:, ::step, ::step].transpose(1, 2, 0).copy()
    over = rgb.copy(); over[find_boundaries(lab[sl], mode='inner')] = (255, 255, 0)
    fig, axes = plt.subplots(1, 4, figsize=(28, 7))
    for ax, (title, arr, cm) in zip(axes, [('image', rgb, None),
                                           (f"extent ('{EXTENT_MODEL}')", np.where(vm[sl], ep[sl], np.nan), 'viridis'),
                                           (f"boundary ('{BOUNDARY_MODEL}', rescaled)",
                                            np.where(vm[sl], normalize_boundary(bp, vm, SEG['BND_NORM'])[sl], np.nan), 'gray'),
                                           ('fields', over, None)]):
        ax.imshow(arr, cmap=cm); ax.set_title(f'{title} (busiest block)'); ax.axis('off')
    plt.tight_layout(); plt.show()
    sample = None

In [ ]:
# 8. EXPORT + PREVIEW
gpkg  = f'{OUT_DIR}/{SITE_NAME}_fields.gpkg'
gjson = f'{OUT_DIR}/{SITE_NAME}_fields.geojson'
bjson = f'{OUT_DIR}/{SITE_NAME}_village_boundary.geojson'
png   = f'{OUT_DIR}/{SITE_NAME}_preview.png'

gdf.to_file(gpkg, driver='GPKG')
gdf.to_crs('EPSG:4326').to_file(gjson, driver='GeoJSON')
gpd.GeoDataFrame({'village': [VILLAGE_NAME], 'source': [boundary_kind]},
                 geometry=[village_geom], crs=UTM_CRS).to_crs('EPSG:4326').to_file(bjson, driver='GeoJSON')

coverage = gdf.area_ha.sum() / village_area_ha * 100
summary = {
    'village': VILLAGE_NAME, 'boundary_source': boundary_desc, 'village_area_ha': round(village_area_ha, 1),
    'n_fields': int(len(gdf)), 'fields_area_ha': round(float(gdf.area_ha.sum()), 1),
    'field_coverage_pct': round(float(coverage), 1), 'median_field_ha': round(float(gdf.area_ha.median()), 3),
    'extent_model': [EXTENT_MODEL, MODELS[EXTENT_MODEL]], 'boundary_model': [BOUNDARY_MODEL, MODELS[BOUNDARY_MODEL]],
    'tta': TTA, 'segmentation': SEG,
    'polygons': {'TOPO_TOL_M': TOPO_TOL_M, 'MAX_VERTICES': MAX_VERTICES, 'MIN_FIELD_M2': MIN_FIELD_M2,
                 'MIN_AREA_FRAC': MIN_AREA_FRAC, 'CLIP_TO_VILLAGE': CLIP_TO_VILLAGE},
    'calibration_best': None if CALIB_BEST is None else {k: CALIB_BEST[k] for k in ('extent', 'boundary', 'setting', 'recall', 'merged', 'split')},
    'source_counts': {k: int(v) for k, v in gdf.source.value_counts().items()},
    'qc_counts': {k: int(v) for k, v in gdf.qc.value_counts().items()},
}
with open(f'{OUT_DIR}/{SITE_NAME}_run_log.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('=' * 60)
print(f"{VILLAGE_NAME}: {len(gdf)} fields | {gdf.area_ha.sum():.0f} ha of {village_area_ha:.0f} ha village ({coverage:.0f}% covered)")
print(f"field size -- median {gdf.area_ha.median():.2f} ha, p95 {gdf.area_ha.quantile(0.95):.2f}, max {gdf.area_ha.max():.2f}")
print(f"vertices per field -- median {gdf.n_vertices.median():.0f}, max {gdf.n_vertices.max()}")
print(f"\n  {gpkg}   (UTM 43N -- QGIS)\n  {gjson}  (WGS84)\n  {bjson}\n  {raster_path}")
print('=' * 60)

step = max(1, int(max(mosaic.shape[1:]) / 3000))
left, top = transform * (0, 0)
right, bottom = transform * (mosaic.shape[2], mosaic.shape[1])
vb = gpd.GeoSeries([village_geom])
fig, axes = plt.subplots(1, 2, figsize=(22, 11))
for ax in axes:
    ax.imshow(mosaic[:, ::step, ::step].transpose(1, 2, 0), extent=(left, right, bottom, top))
    vb.boundary.plot(ax=ax, color='yellow', linewidth=1.2)
    ax.set_xlim(bminx - 50, bmaxx + 50); ax.set_ylim(bminy - 50, bmaxy + 50); ax.axis('off')
axes[0].set_title(f'{VILLAGE_NAME} -- imagery + village boundary')
gdf.boundary.plot(ax=axes[1], color='cyan', linewidth=0.6)
axes[1].set_title(f'{len(gdf)} field polygons')
plt.tight_layout(); plt.savefig(png, dpi=120); plt.show()